In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("EDA").getOrCreate()

df = spark.table("workspace.default.gold_anime")
display(df)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

anime_df = spark.table("workspace.default.gold_anime").toPandas()
print(anime_df.head())

In [0]:
anime_df.shape

**Distribution**<br>
Q1. What does the score distribution look like?

In [0]:
condition = anime_df[anime_df["score"] > 0]
plt.hist(condition["score"], bins=20, color='steelblue')
plt.title("Anime Score Distribution")   
plt.xlabel("Score")                  
plt.ylabel("Number of Anime")           
plt.show()

**Insights**<br>
Most anime scores cluster between 6 and 8,
peaking at 7. This bell curve shape tells us
the "typical anime" is decent but unremarkable.
Scores below 5 are extremely rare — suggesting
either bad anime gets abandoned before enough
people rate it, or MAL users tend to be generous.
Only 8% of anime reach Excellent tier (8.0+),
confirming masterpiece quality is genuinely rare.


**Q2. How skewed are members?**

In [0]:
import numpy as np

#Without log
plt.hist(anime_df["members"], bins=30, color='steelblue')
plt.title("Members Distribution (Raw)")
plt.xlabel("Members")
plt.ylabel("Number of Anime")          
plt.show()

**Insights**<br>
Raw distribution is heavily right skewed —
a tiny number of anime like Naruto and One Piece
have millions of members while most anime
have under 50,000. This "superstar effect"
means a few titles dominate the entire platform.


In [0]:
#With Log
plt.hist(np.log(anime_df["members"]), bins=30, color='steelblue')
plt.title("Members Distribution (Log Scale)")  
plt.xlabel("Log(Members)")
plt.ylabel("Number of Anime")                  
plt.show()

**Insights**<br>
The log transformation reveals a normal
distribution hidden beneath the skew —
confirming why log was the right choice
for our engagement_score formula in Gold layer.
Without log, Naruto or anime with more members would dominate every metric.


**Q3. What is the score tier breakdown?**

In [0]:
tier = anime_df["score_tier"].value_counts()

plt.pie(tier, labels=tier.index,autopct='%1.1f%%')
plt.title("Anime Score Tier Distribution")
plt.show()

**Insights**<br>
Average tier (29.4%) is the largest group —
most anime are watchable but forgettable.
Good tier (28.3%) is close behind showing
a healthy portion of quality content.
Excellent is only 8% — quality is rare.
Unscored at 21% is significant — 1 in 5 anime
has no ratings, mostly recent(ie., stil airing like that) or obscure titles.
Below Average at 13.2% shows the industry
does produce genuinely poor content regularly.

**Trend Over Time**<br>
**Q4. How many anime were released each year?**

In [0]:
filtered_df = anime_df[
    anime_df["year"] < 2026
]

filtered_df.groupby('year')['score']\
    .count()\
    .plot(kind='line')

plt.title("Number of Anime per Year")
plt.xlabel("Year")
plt.ylabel("Number of Anime")
plt.show()


**Insights**<br>
Production grew steadily from ~140 titles in 2010
to a peak of ~280 in 2016-2017 — doubling in
just 7 years. This boom was driven by streaming
platforms like Crunchyroll and Netflix creating
massive international demand. Post 2018 shows
a correction as the market matured. `The dip in
2024-2025 reflects incomplete data for recently
aired anime not yet fully rated on MAL.`

**Q5. Has average score changed over years?**

In [0]:
anime_df[
    (anime_df["score"] > 0) &
    (anime_df["year"] < 2026)    
]\
    .groupby('year')['score']\
    .mean()\
    .plot(kind='line')

plt.title("Average Anime Score by Year")  
plt.xlabel("Year")                         
plt.ylabel("Average Score")               
plt.show()                                 

**Insights**<br>
Scores show a clear dip from 7.0 in 2010
down to 6.6 in 2017 — exactly when production
was at its peak. This is the **quantity kills
quality effect** — `as more anime flooded the
market, average quality dropped.`

The strong recovery to 7.2 by 2023 suggests
streaming platforms began curating content
more selectively, rewarding quality over
volume. `The anime industry self-corrected.`

**Q6. Which season releases most anime?**

In [0]:
grouped_val_2 = anime_df.groupby('season')["title"].count()

plt.bar(grouped_val_2.index, grouped_val_2.values)
plt.xlabel("Season")
plt.ylabel("Number of Anime")
plt.title("Number of Anime per Season")
plt.xticks(rotation=0)    
plt.show()

**Insights**<br>
Spring dominates with 1,100+ releases, followed
by Fall at ~1,000. Summer is consistently weakest.
This mirrors Japan's academic calendar — spring
marks new school terms making it prime time for
studios to launch flagship titles targeting
students. Summer likely suffers as students
are on holiday and away from screens.


**Q7. What are the top 10 most common genres?**

In [0]:
# Define tags to remove — not real genres
non_genres = ['Award Winning', 'Unknown']

genres = (anime_df['genres']
          .str.split(',')
          .explode()
          .str.strip()
          .pipe(lambda x: x[~x.isin(non_genres)])  
          .value_counts()
          .head(10))

plt.bar(genres.index, genres.values)
plt.xlabel("Genres")
plt.xticks(rotation=90)
plt.ylabel("Number of Anime")
plt.title("Top 10 Genres")
plt.show()

**Insights**<br>
Comedy (1,478) and Action (1,172) dominate
by volume — together accounting for nearly
half of all anime. Fantasy (975) is a strong
third. This reflects the core anime demographic:
young male viewers who prefer action-heavy
or lighthearted content. The long tail of
smaller genres (Sports, Suspense, Horror)
shows the industry serves niche audiences too.


**Q8. Which genre has highest average score?**

In [0]:
df_exploded = anime_df.assign(
    genres=anime_df['genres'].str.split(',')
).explode('genres')

df_exploded['genres'] = df_exploded['genres'].str.strip()

df_exploded = df_exploded[
    ~df_exploded['genres'].isin(non_genres) 
]

# Only genres with 50+ anime for fair comparison
genre_counts = df_exploded[
    ~df_exploded['genres'].isin(non_genres)
].groupby('genres')['title'].count()

valid_genres = genre_counts[genre_counts >= 50].index

gen_avgscore = (df_exploded[
    (df_exploded['score'] > 0) &
    (df_exploded['genres'].isin(valid_genres))
    ]
    .groupby('genres')['score']
    .mean()
    .sort_values(ascending=False)
    .head(10))

fig, ax = plt.subplots(figsize=(10, 6))      
ax.barh(gen_avgscore.index,
        gen_avgscore.values,
        color='skyblue')
for i, v in enumerate(gen_avgscore.values):
    ax.text(v / 2, i, f'{v:.2f}',
            va='center', ha='center',
            fontsize=10, fontweight='bold',
            color='white')                   
ax.set_xlabel("Average Score")
ax.set_ylabel("Genres")
ax.set_title("Top 10 Genres by Average Score")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


**Insights**<br>
Narrative driven genres dominate quality:
Drama (~7.3), Suspense (7.2), Mystery (7.14)
all outscore mainstream genres.

Action and Fantasy sit at the bottom of the
top 10 despite being most produced.
This reveals that volume dilutes quality —
the more anime made in a genre, the more
low quality entries drag the average down.

Emotional storytelling consistently beats
spectacle in audience appreciation.


In [0]:
# Check how many anime per genre
df_exploded[~df_exploded['genres'].isin(non_genres)]\
    .groupby('genres')['title']\
    .count()\
    .sort_values(ascending=False)

**Q9. Which genre builds most loyal fans?**

In [0]:
retention = (df_exploded[
    df_exploded['genres'].isin(valid_genres)
    ]
    .groupby('genres')['retention_proxy']
    .mean()
    .sort_values(ascending=False)
    .head(10))

fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(retention.index,
        retention.values,
        color='steelblue')


for i, v in enumerate(retention.values):
    ax.text(v / 2, i, f'{v:.4f}',
            va='center', ha='center',
            fontsize=10, fontweight='bold',
            color='white')

ax.set_xlabel("Retention Proxy")
ax.set_ylabel("Genres")
ax.set_title("Retention Proxy by Genre")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Insights**<br>
Suspense and Drama build the most loyal fans
as measured by favorites/members ratio.
This aligns with Q8 — the same genres that
score highest also retain fans most deeply.

Action despite being most popular genre
shows lower retention — viewers watch
action anime casually without becoming
passionate enough to favorite them.

`Deep emotional investment = loyal fanbase.`<br>
`Casual entertainment = high viewers, low loyalty.`

**Studio Analysis**<br>
**Q10. Which studios produce most anime?**

In [0]:
studio_counts = (anime_df[
    anime_df['studios'] != 'Unknown' 
    ]
    ['studios']
    .value_counts()
    .head(10))

plt.bar(studio_counts.index, studio_counts.values)
plt.xlabel("Studios")
plt.ylabel("Number of Anime")
plt.title("Top 10 Studios by Anime Count")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Insights**<br>
JC Staff leads production volume by a
significant margin, nearly double the
second place studio. This reflects JC Staff's
strategy of producing multiple seasonal anime
simultaneously across different genres.

High volume studios like JC Staff, Studio Deen
and A-1 Pictures prioritize output over
specialization — which likely impacts their
average quality scores in Q11.

**Q11. Which studio has highest average score?**

In [0]:
#Remove Unknown studios
clean_studios = anime_df[
    (anime_df['studios'] != 'Unknown') &
    (anime_df['score'] > 0)
]

#Filter studios with more than 10 anime
studio_counts = clean_studios.groupby('studios')['title'].count()
active_studios = studio_counts[studio_counts >= 10].index

#Calculate average score per studio
studio_avg_score = (clean_studios[
    clean_studios['studios'].isin(active_studios)
    ]
    .groupby('studios')['score']
    .mean()
    .sort_values(ascending=False)
    .head(10))

fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(studio_avg_score.index,
        studio_avg_score.values,
        color='skyblue')

for i, v in enumerate(studio_avg_score.values):
    ax.text(v / 2, i, f'{v:.2f}',  
            va='center', ha='center',
            fontsize=10, fontweight='bold',
            color='white')

ax.set_xlabel("Average Score")
ax.set_ylabel("Studio")
ax.set_title("Top 10 Studios by Average Score")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Insights**<br>
The studios with highest average scores are
NOT the most prolific ones from Q10.
This confirms the volume vs quality tradeoff —
`smaller focused studios consistently outperform
large volume producers in quality metrics.`

Studios that score highest tend to be selective,
taking fewer projects but executing them
at a higher level — a quality over quantity
production philosophy.


**Correlation Analysis**<br>
**Q12. Does higher score mean more members?**

In [0]:
clean = anime_df[anime_df['score'] > 0]

plt.scatter(clean['score'],np.log(clean['members']),alpha=0.3,s=10)
plt.xlabel('Score')
plt.ylabel('Log Members')
plt.title('Anime Score vs Members')
plt.show()
            

**Insights**<br>
There is a clear positive trend — higher scored
anime attract more members overall. However
the relationship has significant scatter,
meaning quality alone does not guarantee
popularity. Some low scored anime have massive
memberships (franchise loyalty, fanservice)
while some high scored anime remain niche
(genre specificity, sequel continuations).

Quality is necessary but not sufficient
for mainstream popularity.


**Q13. What correlates most with score?**

In [0]:
cols = ['score', 'members', 'favorites',
        'engagement_score', 'retention_proxy',
        'popularity_momentum']

corr_matrix = anime_df[cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,       # show numbers
    fmt='.2f',        # 2 decimal places
    cmap='coolwarm',  # red=positive, blue=negative
    center=0
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

**Key findings from correlations:**
- favorites and members correlate strongly
  (people who join also tend to favorite)
- engagement_score correlates with both
  (validates our Gold layer formula - favorites and members)
- score has weak correlation with members
  (quality ≠ popularity confirmed again)
- popularity_momentum is largely independent
  (viral success follows different rules)
- retention_proxy is relatively independent
  (loyalty is driven by different factors
  than raw viewership)


**Q14. Does source material affect score?**

In [0]:
source_score = (anime_df[anime_df['score'] > 0]
                .groupby('source')['score']
                .mean()
                .sort_values(ascending=False)).head(10)


fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(source_score.index,
        source_score.values,
        color='skyblue')

for i, v in enumerate(source_score.values):
    ax.text(v / 2, i, f'{v:.2f}',
            va='center', ha='center',
            fontsize=10, fontweight='bold',
            color='white')     

ax.set_xlabel("Average Score")
ax.set_ylabel("Source")
ax.set_title("Average Score by Source Material")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Insights**<br>
Manga and Light novels adaptations tend to
score higher than original anime. This makes
sense — adapted works have pre-tested story
structures that audiences already validated
through the source material.

Original anime carry more creative risk —
some become masterpieces but many underperform
without the safety net of proven source material.
Game adaptations tend to score lower, possibly
due to difficulty translating interactive
narratives into passive viewing.

**Hidden Insights**<br>
**Q15. Find hidden gem anime**

In [0]:
hidden_gems = (anime_df[
    (anime_df['score_z'] > 1) &          # high quality
    (anime_df['popularity'] > 5000) &    # not mainstream
    (anime_df['scored_by'] >= 1000) &    # reliable ratings
    (anime_df['members'] >= 5000)        # real audience
    ]
    .sort_values('score_z', ascending=False)
    [['title', 'score', 'popularity',
      'members', 'scored_by', 'score_z', 'genres']]
    .head(10))

print("🔍 Hidden Gems — Great anime nobody watched:")
print(hidden_gems.to_string(index=False))

**Insights**<br>
All four hidden gems share common traits:
they are sequels or continuations of niche
shows with small but passionate fanbases.

IDOLiSH7, Chiikawa, Pui Pui Molcar and
Love Live Superstar are all deeply loved
by their communities but never broke into
mainstream consciousness.

Hidden gems are not undiscovered masterpieces
waiting to be found — they are niche productions
perfectly crafted for specific audiences who
appreciate them deeply. Quality exists far
outside the mainstream spotlight.


**Q16. Find overhyped anime**

In [0]:
overhyped = (anime_df[
    (anime_df['score_z'] < -0.5) &       # below average quality
    (anime_df['members'] > 100000) &     # very popular
    (anime_df['scored_by'] >= 1000)      # reliable ratings
    ]
    .sort_values('members', ascending=False)
    [['title', 'score', 'members',
      'scored_by', 'score_z', 'genres']]
    .head(10))

print("⚠️ Overhyped Anime — Popular but poorly rated:")
print(overhyped.to_string(index=False))


**Insights**<br>
Two clear patterns dominate the overhyped list:

Pattern 1 — Sequel Disappointments (top 3):
Tokyo Ghoul:re, Neverland S2, Boruto all
had beloved predecessors. Franchise loyalty
kept memberships in the hundreds of thousands
while quality crashed dramatically.
Neverland fell from 8.7 to 5.25 — the largest
quality drop in the dataset.

Pattern 2 — Fanservice Shows (remaining 7):
Ecchi and romance titles attract large casual
audiences but score poorly critically.
Members join for content appeal not story quality.

Franchise power and fanservice are the two
reliable drivers of popularity regardless
of quality — a sustainable short term but
reputationally damaging long term strategy.

**Q17. Which era produced best quality anime?**

In [0]:
era_order = [
    'Modern Renaissance',    # ← reversed order
    'Global Expansion Era',  # because barh plots
    'Streaming Revolution',  # bottom to top
    'Digital Boom Era'       # so bottom = first era
]

era_quality = (anime_df[anime_df['score_z'].notna()]
               .groupby('airing_era')['score_z']
               .mean()
               .reindex(era_order))

colors = ['#2ecc71' if v >= 0 else '#e74c3c' 
          for v in era_quality.values]

fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(era_quality.index,
        era_quality.values,
        color=colors,
        height=0.5)

ax.axvline(x=0, color='black', linewidth=0.8)

for i, v in enumerate(era_quality.values):
    offset = 0.008 if v >= 0 else -0.008
    align  = 'left' if v >= 0 else 'right'
    ax.text(v + offset, i, f'{v:.3f}',
            va='center', ha=align, 
            fontsize=11, fontweight='bold')

ax.set_xlabel("Average Z-Score")
ax.set_title("Anime Quality by Era",
             fontsize=14, fontweight='bold')

ax.set_xlim(-0.22, 0.22)

from matplotlib.patches import Patch
legend = [
    Patch(color='#2ecc71', label='Above Average'),
    Patch(color='#e74c3c', label='Below Average')
]
ax.legend(handles=legend, loc='lower left')

plt.tight_layout()
plt.show()

**Insights**<br>
The era quality chart tells a surprising story.
Counterintuitively the oldest era (Digital Boom)
and newest era (Modern Renaissance) both score
above average (+0.14), while the middle eras
score below average.

The streaming revolution (2013-2016) actually
hurt quality by incentivising volume production.
The global expansion era (2017-2020) was the
worst for quality as studios rushed content
for international streaming platforms.

Post 2021 the industry self-corrected —
platforms started competing on prestige content
rather than volume, driving quality back up.
More content is not always better content.

**5 Key Findings from Anime Industry Analysis (2010–2025)**

**1. Excellence is Rare but Recovering**
Only 8% of anime achieve Excellent ratings (8.0+), yet the Modern Renaissance era (2021–2025) shows the highest quality scores in the dataset — `proving the industry is producing less but better content.`


**2. Volume Dilutes Quality**
Action and Comedy are the most produced genres but rank lowest in average score. Drama, Suspense and Mystery produce far fewer titles yet consistently score highest. `Restraint in production drives quality up.`


**3. Franchise Loyalty Overrides Quality**
Disappointing sequels dominate the overhyped list. Neverland Season 2 dropped from 8.7 to 5.25 yet kept nearly 1 million members. `Studios exploit franchise loyalty to guarantee large audiences regardless of content quality.`


**4. Streaming First Hurt Then Improved Quality**
Demand from Netflix and Crunchyroll pushed studios toward volume over craft between 2013–2020, `dropping average quality scores. Post 2021, platforms shifted toward prestige content — driving the quality recovery visible today.`

**5. The Best Anime Rarely Reach Mainstream**
Hidden gems like Chiikawa and Pui Pui Molcar score above 8.0 with reliable ratings yet remain niche. This reflects two realities — `some anime are deliberately built for specific audiences with niche tastes, and format experimentation like short-form animation naturally limits mainstream reach. `Outstanding quality exists across the catalogue, but audience specificity keeps the best content invisible to casual viewers.